In [ ]:
from pathlib import Path
from google.colab import drive

drive.mount("/content/drive")

drive_root = Path("/content/drive/MyDrive/vietnamese-summarization")

for folder_name in ["data", "checkpoints", "outputs"]:
    folder_path = drive_root / folder_name

    if not folder_path.is_dir():
        raise RuntimeError(f"Lỗi: Chưa có thư mục Drive: {folder_path}")

print("Đã kết nối Drive và tìm thấy đủ thư mục dự án.")

In [ ]:
import os
import sys
import subprocess
from pathlib import Path

repo_url = "https://github.com/MD1809/vietnamese-summarization.git"
repo_dir = Path("/content/vietnamese-summarization")
target_commit = "bd92efd65cd21ed2add88defd3a110e931f5d573"

if not repo_dir.exists():
    subprocess.run(
        ["git", "clone", repo_url, str(repo_dir)],
        check=True,
    )

if not (repo_dir / ".git").is_dir():
    raise RuntimeError("Lỗi: Thư mục dự án không phải Git repository.")

remote_result = subprocess.run(
    ["git", "remote", "get-url", "origin"],
    cwd=repo_dir,
    capture_output=True,
    text=True,
    check=True,
)

if remote_result.stdout.strip() != repo_url:
    raise RuntimeError("Lỗi: Repository có origin khác với dự án.")

status_result = subprocess.run(
    ["git", "status", "--porcelain"],
    cwd=repo_dir,
    capture_output=True,
    text=True,
    check=True,
)

if status_result.stdout.strip() != "":
    raise RuntimeError(
        "Lỗi: Repository có thay đổi chưa lưu. "
        "Cần kiểm tra trước khi chuyển commit."
    )

subprocess.run(
    ["git", "fetch", "origin"],
    cwd=repo_dir,
    check=True,
)

subprocess.run(
    ["git", "checkout", "--detach", target_commit],
    cwd=repo_dir,
    check=True,
)

commit_result = subprocess.run(
    ["git", "rev-parse", "HEAD"],
    cwd=repo_dir,
    capture_output=True,
    text=True,
    check=True,
)

git_commit = commit_result.stdout.strip()

os.chdir(repo_dir)

if str(repo_dir) not in sys.path:
    sys.path.insert(0, str(repo_dir))

print("Commit đang sử dụng:", git_commit)
print("Thư mục làm việc:", Path.cwd())

In [ ]:
%pip uninstall -y diffusers
%pip install "jedi==0.20.1"
%pip install -r /content/vietnamese-summarization/requirements.txt
%pip check

In [ ]:
import importlib
import json
import platform
from importlib.metadata import version

packages = {
    "torch": "torch",
    "transformers": "transformers",
    "datasets": "datasets",
    "sentencepiece": "sentencepiece",
    "rouge-score": "rouge_score",
    "pandas": "pandas",
    "matplotlib": "matplotlib",
    "gradio": "gradio",
    "numpy": "numpy",
}

library_versions = {}

for package_name in packages:
    module_name = packages[package_name]
    importlib.import_module(module_name)
    library_versions[package_name] = version(package_name)

result = {
    "python_version": platform.python_version(),
    "library_versions": library_versions,
    "import_check_passed": True,
}

print(json.dumps(result, ensure_ascii=False, indent=2))

In [ ]:
import json
import platform
import torch
from datetime import datetime, timezone

environment = {
    "run_id": "colab_environment_check",
    "checked_at_utc": datetime.now(timezone.utc).isoformat(),
    "python_version": platform.python_version(),
    "pytorch_version": str(torch.__version__),
    "pytorch_cuda_version": torch.version.cuda,
    "cuda_available": torch.cuda.is_available(),
    "gpus": [],
    "gpu_check_passed": False,
    "diagnostics": [],
}

if environment["cuda_available"]:
    for index in range(torch.cuda.device_count()):
        gpu = torch.cuda.get_device_properties(index)

        gpu_info = {
            "index": index,
            "name": gpu.name,
            "total_vram_gib": round(gpu.total_memory / 1024**3, 2),
        }

        environment["gpus"].append(gpu_info)

    if len(environment["gpus"]) > 0:
        environment["gpu_check_passed"] = True
else:
    environment["diagnostics"].append(
        "Chưa truy cập được GPU. Hãy kiểm tra loại runtime."
    )

print(json.dumps(environment, ensure_ascii=False, indent=2))

if not environment["gpu_check_passed"]:
    raise RuntimeError("Lỗi: Kiểm tra môi trường GPU chưa đạt.")

In [ ]:
import json

if "environment" not in globals():
    raise RuntimeError("Lỗi: Chưa có dữ liệu môi trường.")

if "drive_root" not in globals():
    raise RuntimeError("Lỗi: Chưa kết nối Drive.")

if "git_commit" not in globals():
    raise RuntimeError("Lỗi: Chưa xác định commit đang sử dụng.")

if "library_versions" not in globals():
    raise RuntimeError("Lỗi: Chưa kiểm tra phiên bản thư viện.")

environment["git_commit"] = git_commit
environment["library_versions"] = library_versions

report_path = drive_root / "outputs" / environment["run_id"] / "config.json"
report_path.parent.mkdir(parents=True, exist_ok=True)

report_path.write_text(
    json.dumps(environment, ensure_ascii=False, indent=2),
    encoding="utf-8",
)

print("Đã lưu hồ sơ trực tiếp trên Drive:", report_path)

In [ ]:
import json
import torch
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
from src.utils import load_train_config

if not torch.cuda.is_available():
    raise RuntimeError("Lỗi: GPU chưa khả dụng để tải mô hình lên CUDA.")

train_config = load_train_config()

model_id = train_config["model"]["id"]
model_revision = train_config["model"]["revision"]

tokenizer = AutoTokenizer.from_pretrained(
    model_id,
    revision=model_revision,
    use_fast=False,
)

sample_text = "Rửa sạch hạt, ngâm qua đêm rồi gieo vào đất ẩm."

encoded_text = tokenizer(
    sample_text,
    return_tensors="pt",
)

token_ids = encoded_text["input_ids"][0]

decoded_text = tokenizer.decode(
    token_ids.tolist(),
    skip_special_tokens=True,
)

model = AutoModelForSeq2SeqLM.from_pretrained(
    model_id,
    revision=model_revision,
)

model = model.to("cuda:0")
model.eval()

model_device = next(model.parameters()).device

if model_device.type != "cuda":
    raise RuntimeError("Lỗi: Mô hình chưa được chuyển lên GPU.")

parameter_count = 0

for parameter in model.parameters():
    parameter_count = parameter_count + parameter.numel()

result = {
    "model_id": model_id,
    "model_revision": model_revision,
    "model_class": type(model).__name__,
    "tokenizer_class": type(tokenizer).__name__,
    "parameter_count": parameter_count,
    "model_device": str(model_device),
    "sample_text": sample_text,
    "token_ids": token_ids.tolist(),
    "decoded_text": decoded_text,
    "model_loading_check_passed": True,
}

print(json.dumps(result, ensure_ascii=False, indent=2))